# Job Chunking and Embeddings

This notebook loads the cleaned job dataset, measures token lengths,
creates overlapping chunks, and generates embeddings.

Prerequisite: run 01_data_exploration.ipynb to generate
data/processed/jobs_cleaned.jsonl.

Initial embedding model: intfloat/multilingual-e5-small.
Maximum model input: 512 tokens, including prefixes and special tokens.

CUDA is used for encoding. Run cells in order; the vector array and chunk table must retain matching row order.

## 1. Load the tokenizer

In [1]:
from pathlib import Path


import pandas as pd

from transformers import AutoTokenizer

embedding_model_name = "intfloat/multilingual-e5-small"

tokenizer = AutoTokenizer.from_pretrained(
    embedding_model_name,
    use_fast=True
)


print("Tokenizer loaded:", embedding_model_name)


print("Tokenizer input limit:", tokenizer.model_max_length)

Tokenizer loaded: intfloat/multilingual-e5-small
Tokenizer input limit: 512


## 2. Load cleaned jobs

In [2]:
project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

cleaned_path = project_root / "data" / "processed" / "jobs_cleaned.jsonl"

df = pd.read_json(cleaned_path, lines=True, convert_dates=False)

print("Loaded shape:", df.shape)

Loaded shape: (1000, 10)


## 3. Measure description tokens

In [3]:
def count_tokens(text):
    token_ids = tokenizer.encode(
        text,
        add_special_tokens=False,
        truncation=False,
        verbose=False
    )

    return len(token_ids)

description_token_counts = df["Clean Description"].apply(count_tokens)

display(description_token_counts.describe())

over_limit_count = (description_token_counts > 512).sum()

print("Descriptions exceeding 512 tokens:", over_limit_count)

count    1000.000000
mean     1098.341000
std       387.406883
min       174.000000
25%       851.000000
50%      1063.000000
75%      1361.250000
max      3266.000000
Name: Clean Description, dtype: float64

Descriptions exceeding 512 tokens: 947


## Token lengths

Descriptions average 1,098.3 tokens, ranging from 174 to 3,266. Of 1,000 jobs,
947 exceed 512 content tokens, so the full descriptions cannot be embedded at once.
Start with 384-token chunks and 48-token overlap, leaving room for the metadata
header, passage prefix, and special tokens. These settings have not been optimized.


## 4. Chunk settings and extraction

In [4]:
MAX_INPUT_TOKENS = 512

CHUNK_SIZE = 384

CHUNK_OVERLAP = 48

assert 0 <= CHUNK_OVERLAP < CHUNK_SIZE

print("Chunk step:", CHUNK_SIZE - CHUNK_OVERLAP)

Chunk step: 336


In [5]:
def chunk_description(text):
    encoded = tokenizer(
        text,
        add_special_tokens=False,
        truncation=False,
        return_offsets_mapping=True,
        verbose=False
    )

    offsets = encoded["offset_mapping"]
    chunks = []
    step = CHUNK_SIZE - CHUNK_OVERLAP

    for start in range(0, len(offsets), step):
        end = min(start + CHUNK_SIZE, len(offsets))

        char_start = offsets[start][0]
        char_end = offsets[end - 1][1]

        chunk_text = text[char_start:char_end]
        chunks.append(chunk_text)

        if end == len(offsets):
            break

    return chunks

### Inspect a sample

In [6]:
sample_job = df.iloc[0]
sample_chunks = chunk_description(sample_job["Clean Description"])

print("Job ID:", sample_job["ID"])
print("Number of chunks:", len(sample_chunks))
print("\nFirst chunk:\n")
print(sample_chunks[0])

Job ID: LF0001
Number of chunks: 3

First chunk:

Job Description:
At Bank of America, we are guided by a common purpose to help make financial lives better through the power of every connection. We do this by driving Responsible Growth and delivering for our clients, teammates, communities and shareholders every day.
Being a Great Place to Work is core to how we drive Responsible Growth. This includes our commitment to being an inclusive workplace, attracting and developing exceptional talent, supporting our teammates' physical, emotional, and financial wellness, recognizing and rewarding performance, and how we make an impact in the communities we serve.
Bank of America is committed to an in-office culture with specific requirements for office-based attendance and which allows for an appropriate level of flexibility for our teammates and businesses based on role-specific considerations.
At Bank of America, you can build a successful career with opportunities to learn, grow, and make 

In [7]:
for chunk_number, chunk_text in enumerate(sample_chunks, start=1):
    token_count = count_tokens(chunk_text)
    print(f"Chunk {chunk_number}: {token_count} content tokens")

print("\nBeginning of the second chunk:\n")
print(sample_chunks[1][:500])

Chunk 1: 384 content tokens
Chunk 2: 384 content tokens
Chunk 3: 114 content tokens

Beginning of the second chunk:

planning related to setting quantitative work priorities in line with the bank's overall strategy and prioritization
Identifies continuous improvements through reviews of approval decisions on relevant model development or model validation tasks, critical feedback on technical documentation, and effective challenges on model development/validation
Maintains and provides oversight of model development and model risk management in respective focus areas to support business requirements and the ent


## 5. Build chunk records

In [8]:
chunk_records = []

for _, job in df.iterrows():
    job_chunks = chunk_description(job["Clean Description"])

    for chunk_index, chunk_text in enumerate(job_chunks):
        record = job.drop(
            labels=["Job Description", "Clean Description"]
        ).to_dict()

        record["chunk_id"] = f"{job['ID']}_chunk_{chunk_index:03d}"
        record["chunk_index"] = chunk_index
        record["chunk_text"] = chunk_text
        record["content_token_count"] = count_tokens(chunk_text)

        chunk_records.append(record)

chunks_df = pd.DataFrame(chunk_records)

print("Total chunks:", len(chunks_df))
print("Jobs represented:", chunks_df["ID"].nunique())

display(
    chunks_df[
        ["chunk_id", "ID", "Job Title", "content_token_count"]
    ].head()
)

Total chunks: 3618
Jobs represented: 1000


,chunk_id,ID,Job Title,content_token_count
0,LF0001_chunk_000,LF0001,"DIR, Equities Quant",384
1,LF0001_chunk_001,LF0001,"DIR, Equities Quant",384
2,LF0001_chunk_002,LF0001,"DIR, Equities Quant",114
3,LF0002_chunk_000,LF0002,Lead Administrator - L1,384
4,LF0002_chunk_001,LF0002,Lead Administrator - L1,384


### Check identifiers, text, and coverage

In [9]:
duplicate_chunk_ids = chunks_df["chunk_id"].duplicated().sum()

empty_chunks = (
    chunks_df["chunk_text"].fillna("").str.strip().eq("").sum()
)

missing_job_ids = set(df["ID"]) - set(chunks_df["ID"])

print("Duplicate chunk IDs:", duplicate_chunk_ids)
print("Empty chunks:", empty_chunks)
print("Jobs without chunks:", len(missing_job_ids))

display(chunks_df["content_token_count"].describe())

assert duplicate_chunk_ids == 0
assert empty_chunks == 0
assert not missing_job_ids

print("Basic chunk checks passed.")

Duplicate chunk IDs: 0
Empty chunks: 0
Jobs without chunks: 0


count    3618.000000
mean      338.393588
std        89.282217
min        49.000000
25%       354.250000
50%       384.000000
75%       384.000000
max       385.000000
Name: content_token_count, dtype: float64

Basic chunk checks passed.


## 6. Build contextual embedding inputs

In [10]:
def build_embedding_text(row):
    return (
        f"passage: Job title: {row['Job Title']}\n"
        f"Company: {row['Company Name']}\n"
        f"Category: {row['Job Category']}\n"
        f"Level: {row['Job Level']}\n\n"
        f"{row['chunk_text']}"
    )

chunks_df["embedding_text"] = chunks_df.apply(
    build_embedding_text,
    axis=1
)

print(chunks_df.iloc[0]["embedding_text"])

passage: Job title: DIR, Equities Quant
Company: Merrill
Category: Data and Analytics
Level: Mid Level

Job Description:
At Bank of America, we are guided by a common purpose to help make financial lives better through the power of every connection. We do this by driving Responsible Growth and delivering for our clients, teammates, communities and shareholders every day.
Being a Great Place to Work is core to how we drive Responsible Growth. This includes our commitment to being an inclusive workplace, attracting and developing exceptional talent, supporting our teammates' physical, emotional, and financial wellness, recognizing and rewarding performance, and how we make an impact in the communities we serve.
Bank of America is committed to an in-office culture with specific requirements for office-based attendance and which allows for an appropriate level of flexibility for our teammates and businesses based on role-specific considerations.
At Bank of America, you can build a successf

### Check the complete input budget

In [11]:
def count_input_tokens(text):
    return len(tokenizer.encode(text, truncation=False))

chunks_df["input_token_count"] = chunks_df["embedding_text"].apply(
    count_input_tokens
)

display(chunks_df["input_token_count"].describe())

oversized_chunks = chunks_df[
    chunks_df["input_token_count"] > MAX_INPUT_TOKENS
]

print("Inputs exceeding 512 tokens:", len(oversized_chunks))

count    3618.000000
mean      366.953289
std        89.533993
min        71.000000
25%       385.000000
50%       410.000000
75%       414.000000
max       438.000000
Name: input_token_count, dtype: float64

Inputs exceeding 512 tokens: 0


### Metadata length

Count the header separately to spot large fields. The final limit check still
uses the complete embedding input.


In [12]:
def count_metadata_tokens(row):
    metadata_only = row.copy()
    metadata_only["chunk_text"] = ""

    metadata_text = build_embedding_text(metadata_only)
    return count_input_tokens(metadata_text)

chunks_df["metadata_token_count"] = chunks_df.apply(
    count_metadata_tokens,
    axis=1
)

display(
    chunks_df.loc[
        chunks_df["input_token_count"] > MAX_INPUT_TOKENS,
        [
            "chunk_id",
            "Job Title",
            "Job Location",
            "content_token_count",
            "metadata_token_count",
            "input_token_count"
        ]
    ].sort_values("input_token_count", ascending=False).head(10)
)

,chunk_id,Job Title,Job Location,content_token_count,metadata_token_count,input_token_count


In [13]:
assert chunks_df["input_token_count"].le(MAX_INPUT_TOKENS).all()

print("All embedding inputs fit within 512 tokens.")

All embedding inputs fit within 512 tokens.


## Full input check

There are 3,618 chunks covering all 1,000 jobs. Full location lists pushed 35
inputs over 512 tokens, with a maximum of 740. Remove location from embedding
text, but keep it in metadata for keyword search and explicit filtering.

The remaining header contains title, company, category, and level. With the
passage prefix and chunk text, all complete inputs fit: the maximum is 438 tokens.
Vector similarity alone does not enforce a location constraint.


## 7. Load the GPU embedding model

In [14]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    embedding_model_name,
    device="cuda"
)

print("Model device:", embedding_model.device)
print("Model input limit:", embedding_model.max_seq_length)

W1028 13:53:24.661000 12152 Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Could not reach the Hub (Cannot reach https://huggingface.co/api/models/intfloat/multilingual-e5-small: offline mode is enabled. To disable it, please unset the `HF_HUB_OFFLINE` environment variable.). Using cached commit hash for 'intfloat/multilingual-e5-small'.


Model device: cuda:0
Model input limit: 512


### Small-batch encoding check

In [15]:
sample_texts = chunks_df["embedding_text"].head(3).tolist()

sample_embeddings = embedding_model.encode(
    sample_texts,
    batch_size=3,
    normalize_embeddings=True,
    convert_to_numpy=True,
    show_progress_bar=False
)

print("Embedding shape:", sample_embeddings.shape)

Embedding shape: (3, 384)


## 8. Encode all chunks

In [16]:
embedding_texts = chunks_df["embedding_text"].tolist()

print("Chunks to encode:", len(embedding_texts), flush=True)

embeddings = embedding_model.encode(
    embedding_texts,
    batch_size=16,
    normalize_embeddings=True,
    convert_to_numpy=True,
    show_progress_bar=False
)

print("Embedding array shape:", embeddings.shape)

Chunks to encode: 3618


Embedding array shape: (3618, 384)


### Validate vector shape and normalization

In [17]:
import numpy as np

expected_shape = (
    len(chunks_df),
    embedding_model.get_embedding_dimension()
)

assert embeddings.shape == expected_shape
assert np.isfinite(embeddings).all()

vector_norms = np.linalg.norm(embeddings, axis=1)

assert np.allclose(vector_norms, 1.0, atol=1e-5)

print("Embedding checks passed.")
print("Minimum vector norm:", vector_norms.min())
print("Maximum vector norm:", vector_norms.max())

Embedding checks passed.
Minimum vector norm: 0.99999994
Maximum vector norm: 1.0000001


## 9. Save aligned vectors and chunk records

In [18]:
processed_dir = project_root / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

embeddings_path = processed_dir / "job_embeddings.npy"
chunks_path = processed_dir / "job_chunks.jsonl"

np.save(embeddings_path, embeddings, allow_pickle=False)

chunks_df.to_json(
    chunks_path,
    orient="records",
    lines=True,
    force_ascii=False
)

print("Saved embeddings:", embeddings_path)
print("Saved chunk records:", chunks_path)

Saved embeddings: C:\Users\NIHAL\Desktop\genai_assignment\data\processed\job_embeddings.npy
Saved chunk records: C:\Users\NIHAL\Desktop\genai_assignment\data\processed\job_chunks.jsonl


### Verify the saved artifacts

In [19]:
loaded_embeddings = np.load(embeddings_path, allow_pickle=False)

loaded_chunks = pd.read_json(
    chunks_path,
    lines=True,
    convert_dates=False
)

np.testing.assert_array_equal(embeddings, loaded_embeddings)

pd.testing.assert_frame_equal(chunks_df.isna(), loaded_chunks.isna())

pd.testing.assert_frame_equal(
    chunks_df.fillna(""),
    loaded_chunks.fillna("")
)

assert len(loaded_chunks) == loaded_embeddings.shape[0]

print("Reload verification passed.")
print("Embedding shape:", loaded_embeddings.shape)
print("Chunk records:", len(loaded_chunks))

Reload verification passed.
Embedding shape: (3618, 384)
Chunk records: 3618
